In [ ]:
import sympy as sp
import numpy as np

In [ ]:
"""
 ---- OPERATOR STRUCTURES ----
 The following operators need to be built into the program, divided into separate categories.
 Spin-1:
 1. Dipolar subspace: {S0, Sx, Sy, Sz}
 2. Real subspace: {S0, Sx, Sz, (Sx)^2, (Sz)^2, {Sx,Sz} }
 3. Full Hermitian subspace: {S0, Sx, Sy, Sz, (Sx)^2, (Sz)^2, {Sx,Sz}, {Sx,Sy}, {Sy,Sz}}
"""
im = sp.I
S0 = sp.eye(3)
Sx = sp.Matrix([[0, 1, 0,], [1, 0, 1], [0, 1, 0]])
Sy = im * sp.Matrix([[0, -1, 0,], [1, 0, -1], [0, 1, 0]])
Sz = sp.Matrix([[1, 0, 0,], [0, 0, 0], [0, 0, -1]])
Sp = Sx + im * Sy
Sm = Sx - im * Sy

# every spin 1 operator that 
SPIN1 = {'0': S0, 'y': Sy, 'x': Sx, 'z': Sz, '+': Sp, '-': Sm,
         'xx': Sx*Sx, 'zz': Sz*Sz, 'xz': Sx*Sz + Sz*Sx,
         'xy': Sx*Sy + Sy*Sx, 'yz': Sy*Sz + Sz*Sy}

# projectors onto the different basis states
PROJECTORS = dict()
for i in range(3):
    diags = [0, 0, 0]
    diags[i] = 1
    PROJECTORS[f'{i}'] = sp.diag(*diags)

# these are the list over which iterations will be carried out
LABELS = ['0', 'x', 'y', 'z']
LABELS_REAL = ['0', 'x', 'z', 'xx', 'zz', 'xz']
LABELS_FULL = ['0', 'x', 'y', 'z', 'xx', 'zz', 'xz', 'xy', 'yz']

# Pauli matrices for spin-1/2 validations
# this uses the same labels as the default spin-1 operators
PAULI = {'0': sp.eye(2), 'x': sp.Matrix([[0, 1], [1, 0]]), 'y': im * sp.Matrix([[0, -1], [1, 0]]), 'z': sp.Matrix([[1, 0], [0, -1]])}


In [ ]:
"""
 ---- ANSATZ GENERATION ----
 This module is responsible for generating the IRF-type ansatz. 
"""

In [ ]:
import itertools as it

for i, j in it.product(range(5), repeat=2):
    # print(i, j)
    pass

In [2]:
import sympy as sp
import numpy as np
from operators import SPIN1, PAULI, LABELS, LABELS_DIP_REAL, LABELS_FULL, LABELS_REAL, PROJECTORS
import itertools as it

In [ ]:
class Ansatz:
    # hdict = {(a, b): some matrix}
    def __init__(self, hdict, d=3, check_hermitian=True):
        self.hdict = {ab: sp.Matrix(M) for ab, M in hdict.items()}
        self.d = d
        syms = set()
        hkeys = self.hdict.keys()
        for a, b in it.product(range(d), repeat=2):
            if (a, b) not in hkeys:
                self.hdict[(a, b)] = sp.zeros(d, d)
            else: 
                m_ab = sp.Matrix(self.hdict[(a, b)])
                if m_ab != 0:
                    for i, j in it.product(range(d), repeat=2):
                        syms_ij = m_ab[i, j].free_symbols()
                        syms = syms.union(syms_ij)
        for M in hdict.values():
            syms = syms.union(*[ij.free_symbols for ij in M])
        self.syms = sorted(syms, key=lambda s: s.name)
        for s in self.syms:
            if not s.is_real:
                raise ValueError(f'the symbol {s} is not real')
        if check_hermitian:
            # this checks whether all blocks specified in hdict are Hermitian (the ansatz is forced to ensure hermiticity by declaring all blocks in hdict Hermitian)
            for a, b in it.product(range(d), repeat=2):
                M = sp.Matrix(hdict[(a, b)])
                if sp.simplify(M - M.adjoint()) != sp.zeros(d, d):
                    raise ValueError(f'the block {a, b} is not Hermitian, the matrix is: {M}')
        self.linear = {}
        sym_names = {s: n for n, s in self.syms.items()}
        for (a, b), M in self.hdict.items():
            for i, j in it.product(range(self.d), repeat=2):
                eq = sp.expand(M[i, j])
                if eq == 0:
                    continue
                poly = sp.Poly(eq, *self.syms) if len(self.syms) != 0 else None
                if poly is not None and poly.total_degree() > 1:
                    raise ValueError(f'slot {(i, j)} in matrix {M} has degree {poly.total_degree()}')
                const = eq.subs({s: 0 for s in eq.free_symbols})
                lin_coeffs = {sym_names[s]: sp.nsimplify(sp.diff(eq, s)) for s in eq.free_symbols if sp.diff(eq, s) != 0}
                self.linear[(a, b, i, j)] = (sp.nsimplify(const), lin_coeffs)
        self.check_rational()

    def check_rational(self):
        for (a, b, i, j), (c, lin_dict) in self.linear.items():
            for x in [c, *lin_dict.values()]:
                re, im = sp.re(x), sp.im(x)
                if not (re.is_Rational and im.is_Rational):
                    raise ValueError(f'a coefficient of M_{i}{j} in block {(a, b)} is not rational')
        return None

    def numeric_block(self, sym_vals):
        num_hdict = dict()
        for (a, b), M in self.hdict.items():
            for s in M.free_symbols:
                num_hdict[(a, b)] = np.array(M.subs(s, sym_vals[s]).evalf(), dtype=complex)
        return num_hdict

def hermitian_block(a, b, d=3):
    hblock = sp.zeros(d, d)
    for i, j in it.product(range(d), repeat=2):
        if i==j:
            u_ii = sp.symbols(f'u_{i}{i}^{a}{b}', real=True)
            hblock += u_ii * PROJECTORS[(i, i)]
        elif i < j:
            u_ij, v_ij = sp.symbols([f'u_{i}{j}^{a}{b}', f'v_{i}{j}^{a}{b}'], real=True)
            hblock += (u_ij + sp.I *v_ij) * PROJECTORS[(i, j)] + (u_ij - sp.I *v_ij) * PROJECTORS[(j, i)]
    return hblock

def block_from_labels(a, b, labels, prefix='A', basis=SPIN1):
    d, _ = basis[labels[0]].shape
    syms = {l: sp.Symbol(f'{prefix}_{a}{b}^{l}', real=True) for l in labels}
    M = sp.zeros(d, d)
    for l in labels:
        M += syms[l] * basis[l]
    return M

def hdict_builder(family='full', d=3, reflect=False, pattern=None, prefix='A'):
    fam_labels = {'full': LABELS_FULL, 'real': LABELS_REAL, 'dipolar': LABELS, 'dipolar_real': ['0', 'x', 'z']}
    basis = SPIN1
    hdict = {}
    for a, b in it.product(range(3), repeat=2):
        (ka, kb) = (min(a, b), max(a, b)) if reflect else (a, b)
        if (ka, kb) in hdict.keys():
            hdict[(a, b)] = hdict[(ka, kb)]
            continue
        active = 1 if pattern is None else pattern(a, b)
        if family == 'hermitian':
            M = hermitian_block(ka, kb, d)
            # if active == 0, the pattern does not let the block to contain off-diagonal parts
            if not active:
                M = sp.diag(*[M[i, i] for i in range(d)])
        else:
            labels = fam_labels.get(family)
            # labels = fam_labels[family]
            if not active:
                diag_labels = [l for l in labels if basis[l].is_diagonal()]
                labels = diag_labels
            M = block_from_labels(ka, kb, labels, prefix=prefix, basis=SPIN1)
        hdict[(ka, kb)] = M
        hdict[(a, b)] = M
    return hdict

def restrict(ansatz, linear_restr_eqs, name=None):
    sol = sp.solve(list(linear_restr_eqs), ansatz.syms, dict=True)
    if not sol:
        raise ValueError('inconsistent set of linear restrictions')
    sub = sol[0]
    hdict = {ab: sp.expand(M.subs(sub)) for ab, M in ansatz.hdict.items()}
    return Ansatz(hdict, d=ansatz.d, name=name or ansatz.name + '|restricted')

In [81]:
hdict = hdict_builder(family='full', reflect=False)
display(hdict[(0, 0)])

Matrix([
[    A_00^0 + A_00^xx + A_00^z + A_00^zz, A_00^x + A_00^xz - I*A_00^y - I*A_00^yz,                   A_00^xx - 2*I*A_00^xy],
[A_00^x + A_00^xz + I*A_00^y + I*A_00^yz,                      A_00^0 + 2*A_00^xx, A_00^x - A_00^xz - I*A_00^y + I*A_00^yz],
[                  A_00^xx + 2*I*A_00^xy, A_00^x - A_00^xz + I*A_00^y - I*A_00^yz,     A_00^0 + A_00^xx - A_00^z + A_00^zz]])

In [18]:
import sympy as sp
import numpy as np
from operators import PAULI

def numeric_block(hdict, sym_vals):
    num_hdict = dict()
    for ab, M in hdict.items():
        for s in M.free_symbols:
            num_hdict[ab] = np.array(M.subs(s, sym_vals[s]).evalf(), dtype=complex)
    return num_hdict

x, y = sp.symbols(['x', 'y'])
mat1 = x * PAULI['x']
mat2 = y * PAULI['y']
hdict = {'0': mat1, '1': mat2}
num_hdict = numeric_block(hdict=hdict, sym_vals={x: 0.5, y: 10})
print(num_hdict['1'][1, 0])
print(PAULI['y'][1, 0])
expr = x**2 + y**3 + x * y - 4
expr_num = expr.subs({s: 3 for s in expr.free_symbols})
print(expr_num)
sym_dict = {'x': x, 'y': y}
if 'x' in sym_dict.keys():
    print('majm')
sym_names = {s: n for n, s in sym_dict.items()}
print(type(sym_names[x]))

10j
I
41
majm
<class 'str'>


In [38]:
# projectors onto the different basis states
PROJECTORS = dict()
for i in range(3):
    diags = [0, 0, 0]
    diags[i] = 1
    PROJECTORS[f'{i}'] = sp.diag(*diags)

def block_from_labels(a, b, labels, prefix='A', basis=SPIN1):
    d, _ = basis[labels[0]].shape
    syms = {l: sp.Symbol(f'{prefix}_{a}{b}^{l}', real=True) for l in labels}
    M = sp.zeros(d, d)
    for l in labels:
        M += syms[l] * basis[l]
    return M

def hermitian_block(d=3):
    hblock = sp.zeros(d, d)
    for i, j in it.product(range(d), repeat=2):
        if i==j:
            u_ii = sp.symbols(f'u_{i}{i}', real=True)
            hblock[i, i] = u_ii
        elif i < j:
            u_ij, v_ij = sp.symbols([f'u_{i}{i}', f'u_{i}{i}'], real=True)
            hblock[i, j] = u_ij + sp.I *v_ij
            hblock[j, i] = u_ij - sp.I *v_ij
    return hblock
hblock = hermitian_block()
mat = sp.Matrix([[0, 1 + sp.I], [0, 1]])
display(mat)
display(mat.adjoint())
M = block_from_labels(0, 0, labels='z', prefix='P', basis=SPIN1) + block_from_labels(0, 0, labels='x', prefix='A', basis=SPIN1)
display(M)
display(M.free_symbols)

Matrix([
[0, 1 + I],
[0,     1]])

Matrix([
[    0, 0],
[1 - I, 1]])

Matrix([
[P_00^z, A_00^x,       0],
[A_00^x,      0,  A_00^x],
[     0, A_00^x, -P_00^z]])

{A_00^x, P_00^z}

In [3]:
family='real'
fam_labels = {'full': LABELS_FULL, 'real': LABELS_REAL, 'dipolar': LABELS, 'dipolar_real': ['0', 'x', 'z']}
labels = fam_labels.get(family, family)
print(labels)

['0', 'x', 'z', 'xx', 'zz', 'xz']


In [51]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import sympy as sp
import numpy as np
from ansatz import Ansatz, hermitian_block, block_from_labels, hdict_builder
from operators import LABELS, LABELS_DIP_REAL, LABELS_FULL, LABELS_REAL, PAULI

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [52]:
M = hermitian_block(0, 2, 4)
display(M)

Matrix([
[            u_00^02, u_01^02 + I*v_01^02, u_02^02 + I*v_02^02, u_03^02 + I*v_03^02],
[u_01^02 - I*v_01^02,             u_11^02, u_12^02 + I*v_12^02, u_13^02 + I*v_13^02],
[u_02^02 - I*v_02^02, u_12^02 - I*v_12^02,             u_22^02, u_23^02 + I*v_23^02],
[u_03^02 - I*v_03^02, u_13^02 - I*v_13^02, u_23^02 - I*v_23^02,             u_33^02]])

In [53]:
a, b = 0, 2
hblock = block_from_labels(a, b, LABELS, prefix='S', basis=PAULI)

display(hblock)

hdict = hdict_builder(family='hermitian', reflection=True, pattern=None, prefix='S')
syms = set()
for (a, b), h_ab in hdict.items():
    # print(f'h_{a, b}:')
    # display(h_ab)
    syms_ab = set()
    for e in h_ab:
        syms_ab |= syms_ab.union(e.free_symbols)
    syms |= syms.union(syms_ab)
print(f'n(syms)={len(syms)}, {syms}')


Matrix([
[  S^02_0 + S^02_z, S^02_x - I*S^02_y],
[S^02_x + I*S^02_y,   S^02_0 - S^02_z]])

n(syms)=54, {v_01^02, v_12^11, u_00^00, u_01^12, u_12^12, v_01^00, v_02^22, u_00^12, u_11^22, u_22^02, u_00^01, u_02^02, u_01^01, u_00^11, u_22^00, u_01^22, u_02^12, u_01^02, v_02^01, u_22^22, u_12^22, u_11^12, u_02^11, u_12^01, u_00^02, u_01^11, v_01^11, u_22^11, v_02^02, u_12^00, v_12^22, u_11^01, u_12^02, v_02^00, u_11^11, v_01^12, v_02^12, v_12^12, u_00^22, v_12^02, u_12^11, v_02^11, u_01^00, v_01^22, u_11^00, v_12^01, u_02^22, u_22^01, v_01^01, u_11^02, u_02^00, v_12^00, u_02^01, u_22^12}


In [67]:
hdict = hdict_builder(family='dipolar_real', reflection=True, pattern=None, prefix='S', basis_name='SPIN1')
print(hdict)
# v00 = sp.Symbol(f'v_00', real=True)
# M = hermitian_block(0, 0)
# M[0, 0] = sp.I * v00
# hdict[(0, 1)] = M
ans = Ansatz(hdict, 3)
print(ans.d)
print(ans.nvars)
print(ans.syms)
print(ans.linear)
# print(ans.check_rational())
# print(ans.hdict)
sym_vals = dict()
for (a, b), M in hdict.items():
    for e in M:
        esyms = e.free_symbols
        for s in esyms:
            sym_vals[s] = np.random.uniform(0, 10)
print(ans.numeric_block(sym_vals))

{(0, 0): Matrix([
[S^00_0 + S^00_z, S^00_x,               0],
[         S^00_x, S^00_0,          S^00_x],
[              0, S^00_x, S^00_0 - S^00_z]]), (0, 1): Matrix([
[S^01_0 + S^01_z, S^01_x,               0],
[         S^01_x, S^01_0,          S^01_x],
[              0, S^01_x, S^01_0 - S^01_z]]), (0, 2): Matrix([
[S^02_0 + S^02_z, S^02_x,               0],
[         S^02_x, S^02_0,          S^02_x],
[              0, S^02_x, S^02_0 - S^02_z]]), (1, 0): Matrix([
[S^01_0 + S^01_z, S^01_x,               0],
[         S^01_x, S^01_0,          S^01_x],
[              0, S^01_x, S^01_0 - S^01_z]]), (1, 1): Matrix([
[S^11_0 + S^11_z, S^11_x,               0],
[         S^11_x, S^11_0,          S^11_x],
[              0, S^11_x, S^11_0 - S^11_z]]), (1, 2): Matrix([
[S^12_0 + S^12_z, S^12_x,               0],
[         S^12_x, S^12_0,          S^12_x],
[              0, S^12_x, S^12_0 - S^12_z]]), (2, 0): Matrix([
[S^02_0 + S^02_z, S^02_x,               0],
[         S^02_x, S^02_0,       